In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# ==============================
# 0) 全局配置
# ==============================
START_DATE = "2019-04-01 00:00:00"
END_DATE = "2019-04-30 23:59:59"
TIME_FREQ = "15min"
UTC_TO_LOCAL_HOURS = 8  # ERA5(UTC) -> 北京时间

STATION_ID = "station05"
TARGET_COL = "power"


def find_project_root() -> Path:
    """从当前工作目录向上寻找包含 stations/weather_data 的项目根目录。"""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "stations").exists() and (p / "weather_data").exists():
            return p
    return cwd


PROJECT_ROOT = find_project_root()
PVOD_FILE = PROJECT_ROOT / "stations" / f"{STATION_ID}.csv"
STATION_META_FILE = PROJECT_ROOT / "stations" / "metadata.csv"
ERA5_2D_INSTANT = PROJECT_ROOT / "weather_data" / "instant_2d.nc"
ERA5_2D_ACCUM = PROJECT_ROOT / "weather_data" / "accum_2d.nc"
ERA5_3D_FILE = PROJECT_ROOT / "weather_data" / "instant_3d.nc"
OUTPUT_FILE = PROJECT_ROOT / "artifacts" / f"{STATION_ID}_merged_2019-04.csv"


def detect_time_col(df: pd.DataFrame) -> str:
    for c in ["time", "date_time", "datetime", "DATETIME", "Time", "DateTime"]:
        if c in df.columns:
            return c
    raise KeyError("未找到时间列，请检查 station05.csv 的时间字段名称")


def normalize_time_coord(ds: xr.Dataset) -> xr.Dataset:
    """把不同命名的时间坐标统一成 time。"""
    for c in ["time", "valid_time", "forecast_time", "datetime"]:
        if c in ds.coords or c in ds.dims:
            return ds if c == "time" else ds.rename({c: "time"})
    raise KeyError(f"未找到时间坐标。可用坐标: {list(ds.coords)}")


def get_station_lat_lon(station_id: str) -> tuple[float, float]:
    meta = pd.read_csv(STATION_META_FILE)
    row = meta.loc[meta["Station_ID"].str.lower() == station_id.lower()]
    if row.empty:
        raise ValueError(f"metadata.csv 中未找到站点: {station_id}")
    # metadata 中是 Longitude / Latitude
    lon = float(row.iloc[0]["Longitude"])
    lat = float(row.iloc[0]["Latitude"])
    return lat, lon


def detect_level_col(df: pd.DataFrame) -> str:
    for c in ["level", "pressure_level", "isobaricInhPa"]:
        if c in df.columns:
            return c
    raise KeyError(f"未找到气压层列。可用列: {list(df.columns)}")


STATION_LAT, STATION_LON = get_station_lat_lon(STATION_ID)

print(f"Project root: {PROJECT_ROOT}")
print(f"Station: {STATION_ID}, lat={STATION_LAT:.5f}, lon={STATION_LON:.5f}")
print("配置加载完成")

Project root: /Users/dazhaxie/Desktop/Deye/PVOD_Forecast
Station: station05, lat=38.23550, lon=114.12360
配置加载完成


In [2]:
# ==============================
# 1) 处理 1D PVOD
# ==============================
print("1/5 读取并截取 PVOD 1D 数据...")

df_pvod = pd.read_csv(PVOD_FILE)
time_col = detect_time_col(df_pvod)

df_pvod[time_col] = pd.to_datetime(df_pvod[time_col])
df_pvod = df_pvod.sort_values(time_col).set_index(time_col)

# 截取 2019-04
df_pvod_apr = df_pvod.loc[START_DATE:END_DATE].copy()

# 所有输入特征加 1D_ 前缀，目标列 power 保留原名
rename_map = {c: f"1D_{c}" for c in df_pvod_apr.columns if c != TARGET_COL}
df_pvod_apr = df_pvod_apr.rename(columns=rename_map)

print(f"PVOD shape: {df_pvod_apr.shape}")
print(df_pvod_apr.head(2))

1/5 读取并截取 PVOD 1D 数据...
PVOD shape: (2880, 14)
                     1D_nwp_globalirrad  1D_nwp_directirrad  \
date_time                                                     
2019-04-01 00:00:00              301.91              263.68   
2019-04-01 00:15:00              360.56              319.55   

                     1D_nwp_temperature  1D_nwp_humidity  1D_nwp_windspeed  \
date_time                                                                    
2019-04-01 00:00:00                7.98            24.14              1.46   
2019-04-01 00:15:00                8.29            23.04              2.15   

                     1D_nwp_winddirection  1D_nwp_pressure  1D_lmd_totalirrad  \
date_time                                                                       
2019-04-01 00:00:00                 68.54          1003.42                387   
2019-04-01 00:15:00                 65.86          1003.55                453   

                     1D_lmd_diffuseirrad  1D_lmd_temperature  

In [3]:
# ==============================
# 2) 处理 2D ERA5
# ==============================
print("2/5 处理 ERA5 2D 网格...")

with xr.open_dataset(ERA5_2D_INSTANT) as ds_2d_inst, xr.open_dataset(ERA5_2D_ACCUM) as ds_2d_accu:
    ds_2d = xr.merge([ds_2d_inst, ds_2d_accu], compat="override")

ds_2d = normalize_time_coord(ds_2d)

# ERA5 时间对齐到北京时间
ds_2d = ds_2d.assign_coords(time=ds_2d["time"] + pd.Timedelta(hours=UTC_TO_LOCAL_HOURS))

# 空间插值到站点点位
ds_2d_point = ds_2d.interp(latitude=STATION_LAT, longitude=STATION_LON, method="linear")

# 常见辐射累计量单位换算：J/m^2 -> W/m^2（按小时步长）
if "ssrd" in ds_2d_point.variables:
    ds_2d_point["ssrd"] = ds_2d_point["ssrd"] / 3600.0

# 转成 DataFrame 并重采样
cols_to_drop = [c for c in ["latitude", "longitude", "number", "expver"] if c in ds_2d_point.coords or c in ds_2d_point.variables]
df_2d = ds_2d_point.to_dataframe().drop(columns=cols_to_drop, errors="ignore")
df_2d = df_2d.sort_index().resample(TIME_FREQ).interpolate("time")

df_2d_apr = df_2d.loc[START_DATE:END_DATE].copy()
df_2d_apr.columns = [f"2D_{c}" for c in df_2d_apr.columns]

print(f"ERA5 2D shape: {df_2d_apr.shape}")
print(df_2d_apr.head(2))

2/5 处理 ERA5 2D 网格...
ERA5 2D shape: (2880, 8)
                       2D_u10    2D_v10      2D_d2m      2D_t2m         2D_sp  \
time                                                                            
2019-04-01 00:00:00  1.675705 -0.367242  264.195374  279.762369  98456.725470   
2019-04-01 00:15:00  1.656263 -0.452503  264.521913  279.454519  98465.495907   

                     2D_tcc  2D_ssrd  2D_fdir  
time                                           
2019-04-01 00:00:00     0.0      0.0      0.0  
2019-04-01 00:15:00     0.0      0.0      0.0  


In [4]:
# ==============================
# 3) 处理 3D ERA5（按气压层展开）
# ==============================
print("3/5 处理 ERA5 3D 气压层...")

with xr.open_dataset(ERA5_3D_FILE) as ds_3d:
    ds_3d = ds_3d.load()

ds_3d = normalize_time_coord(ds_3d)

# ERA5 时间对齐到北京时间
ds_3d = ds_3d.assign_coords(time=ds_3d["time"] + pd.Timedelta(hours=UTC_TO_LOCAL_HOURS))

# 空间插值到站点点位
ds_3d_point = ds_3d.interp(latitude=STATION_LAT, longitude=STATION_LON, method="linear")

# 展平 level 维度
df_3d_raw = ds_3d_point.to_dataframe().reset_index()
level_col = detect_level_col(df_3d_raw)
exclude_cols = {"time", level_col, "latitude", "longitude", "number", "expver"}
value_cols = [c for c in df_3d_raw.columns if c not in exclude_cols]

if not value_cols:
    raise ValueError("3D 数据未找到可用变量，请检查 instant_3d.nc")

df_3d_pivot = df_3d_raw.pivot_table(index="time", columns=level_col, values=value_cols, aggfunc="mean")
df_3d_pivot.columns = [f"3D_{var}_{int(lvl)}hPa" for var, lvl in df_3d_pivot.columns]

df_3d_resampled = df_3d_pivot.sort_index().resample(TIME_FREQ).interpolate("time")
df_3d_apr = df_3d_resampled.loc[START_DATE:END_DATE].copy()

print(f"ERA5 3D shape: {df_3d_apr.shape}")
print(df_3d_apr.head(2))

3/5 处理 ERA5 3D 气压层...
ERA5 3D shape: (2848, 12)
                     3D_r_500hPa  3D_r_850hPa  3D_r_1000hPa  3D_t_500hPa  \
time                                                                       
2019-04-01 08:00:00    64.651910    21.271682     35.229239   251.795559   
2019-04-01 08:15:00    66.018754    21.297007     31.967927   251.851198   

                     3D_t_850hPa  3D_t_1000hPa  3D_u_500hPa  3D_u_850hPa  \
time                                                                       
2019-04-01 08:00:00   273.934231    281.633277    21.638643    -0.337639   
2019-04-01 08:15:00   273.879359    281.870928    21.611037    -0.302443   

                     3D_u_1000hPa  3D_v_500hPa  3D_v_850hPa  3D_v_1000hPa  
time                                                                       
2019-04-01 08:00:00      1.120589   -17.030370    -3.851672      0.155003  
2019-04-01 08:15:00      0.667049   -17.002012    -3.553181      0.094933  


In [5]:
# ==============================
# 4) 合并并导出
# ==============================
print("4/5 合并 1D + 2D + 3D 并导出...")

final_df = pd.concat([df_pvod_apr, df_2d_apr, df_3d_apr], axis=1, join="inner").sort_index()

# 删除潜在空值（常见于插值边界）
final_df = final_df.dropna().copy()

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_FILE, index=True)

print(f"导出完成: {OUTPUT_FILE}")
print(f"最终维度: {final_df.shape}")
print(final_df.head(2))

4/5 合并 1D + 2D + 3D 并导出...
导出完成: /Users/dazhaxie/Desktop/Deye/PVOD_Forecast/artifacts/station05_merged_2019-04.csv
最终维度: (2848, 34)
                     1D_nwp_globalirrad  1D_nwp_directirrad  \
2019-04-01 08:00:00              494.58              448.51   
2019-04-01 08:15:00              441.94              397.70   

                     1D_nwp_temperature  1D_nwp_humidity  1D_nwp_windspeed  \
2019-04-01 08:00:00               15.88            16.19              6.30   
2019-04-01 08:15:00               15.85            16.16              6.31   

                     1D_nwp_winddirection  1D_nwp_pressure  1D_lmd_totalirrad  \
2019-04-01 08:00:00                127.46           998.29                497   
2019-04-01 08:15:00                131.78           998.21                437   

                     1D_lmd_diffuseirrad  1D_lmd_temperature  ...  \
2019-04-01 08:00:00                  233           16.900000  ...   
2019-04-01 08:15:00                  218           16.799999 

In [6]:
# ==============================
# 5) 对齐质量检查（为后续特征分析做准备）
# ==============================
print("5/5 对齐质量检查...")

summary = {
    "rows": len(final_df),
    "cols": final_df.shape[1],
    "start": str(final_df.index.min()),
    "end": str(final_df.index.max()),
    "target_exists": TARGET_COL in final_df.columns,
    "target_missing_ratio": float(final_df[TARGET_COL].isna().mean()) if TARGET_COL in final_df.columns else np.nan,
    "freq_guess": str(pd.infer_freq(final_df.index[:200])) if len(final_df) >= 10 else "N/A",
}

print(pd.Series(summary))

# 简单查看与 power 的线性相关性 Top20（仅数值列）
if TARGET_COL in final_df.columns:
    corr = final_df.select_dtypes(include=[np.number]).corr(numeric_only=True)[TARGET_COL].dropna().sort_values(key=np.abs, ascending=False)
    print("\n与 power 相关性 Top20（按绝对值排序）:")
    print(corr.head(20))
else:
    print("警告: 合并后不存在 power 列，请检查 1D 源数据字段")

5/5 对齐质量检查...
rows                                   2848
cols                                     34
start                   2019-04-01 08:00:00
end                     2019-04-30 23:45:00
target_exists                          True
target_missing_ratio                    0.0
freq_guess                            15min
dtype: object

与 power 相关性 Top20（按绝对值排序）:
power                   1.000000
1D_lmd_totalirrad       0.990872
1D_nwp_globalirrad      0.938106
1D_nwp_directirrad      0.929752
1D_lmd_diffuseirrad     0.783581
3D_u_1000hPa            0.519970
1D_nwp_temperature      0.517403
1D_lmd_temperature      0.489783
2D_u10                  0.480461
2D_t2m                 -0.436101
1D_nwp_humidity        -0.422276
2D_ssrd                -0.419417
1D_lmd_windspeed        0.389193
2D_fdir                -0.365369
3D_r_1000hPa            0.346926
1D_lmd_winddirection   -0.222284
1D_nwp_windspeed        0.211641
3D_t_1000hPa           -0.200685
1D_nwp_winddirection   -0.200302
2D_tcc   